# 🧠 Notebook 06 — LangChain RAG Pipeline
**Healthcare RAG-Powered Medical Q&A Assistant**
**eyouth × DEPI | Microsoft Machine Learning Track | 2026**

---

### 🎯 Objectives
- Build a RAG pipeline: embed query → retrieve top-5 → generate answer
- Use `flan-t5-base` as the free local LLM
- Append mandatory medical disclaimer to every response
- Test on 10 sample queries with full logging
- Build and verify `src/rag/pipeline.py`

### 📋 Deliverables
- `notebooks/05_rag_pipeline.ipynb`
- `src/rag/pipeline.py`

---

## 1. Imports & Setup

In [1]:
import os
import sys
import time
import json
import pandas as pd

# Add project root to path
sys.path.append(os.path.abspath('..'))

print("✅ Imports ready")

✅ Imports ready


## 2. Build RAG Pipeline

This imports from `src/rag/pipeline.py` which implements:
1. **Encoder**: `pritamdeka/S-PubMedBert-MS-MARCO` (PubMedBERT domain model)
2. **Vector store**: FAISS `IndexFlatIP`
3. **Generator**: `google/flan-t5-base` (free, local, no API key)
4. **Disclaimer**: Appended to every response automatically

In [2]:
from src.rag.pipeline import build_rag_pipeline

pipeline = build_rag_pipeline()

Loading embedding model: pritamdeka/S-PubMedBert-MS-MARCO
Loading FAISS index: D:\Projects\Healthcare-RAG-Powered-Medical-QA-Assistant\data\embeddings\faiss_index\pubmedqa_index_flatl2.faiss
  Vectors in index: 209,108
Loading chunk mapping: D:\Projects\Healthcare-RAG-Powered-Medical-QA-Assistant\data\embeddings\faiss_index\chunk_mapping.pkl
[OK] BM25 index built over 209,108 documents
📂 Loading classifier from local: D:\Projects\Healthcare-RAG-Powered-Medical-QA-Assistant\models\classifier\biobert_classifier
✅ Classifier loaded | Device: cuda | Classes: ['Diagnosis', 'General', 'Medication', 'Prevention', 'Symptoms', 'Treatment']
[OK] Classifier ready for category routing
Loading reranker: cross-encoder/ms-marco-MiniLM-L-12-v2
[OK] Reranker ready
GROQ_API_KEY not set — falling back to local flan-t5-base
[OK] RAG Pipeline ready


## 3. Test Retrieval (Top-5)

Verify that FAISS returns relevant medical context for a sample query.

In [3]:
test_query = "What are the symptoms of diabetes?"

retrieved = pipeline.retrieve(test_query, top_k=5)

print(f"Query: {test_query}")
print(f"Retrieved {len(retrieved)} chunks:\n")

for i, chunk in enumerate(retrieved):
    print(f"--- Chunk {i+1} | ID: {chunk['chunk_id']} | "
          f"Category: {chunk['category']} | Distance: {chunk['distance']:.4f} ---")
    print(f"Q: {chunk['question'][:100]}")
    print(f"A: {chunk['answer'][:150]}")
    print()

Query: What are the symptoms of diabetes?
Retrieved 5 chunks:

--- Chunk 1 | ID: 201425 | Category: Symptoms | Distance: -12.3967 ---
Q: do symptoms of depression prospectively predict poorer self-care in patients with type 2 diabetes?
A: depressive symptoms predict subsequent non-adherence to important aspects of self-care in patients with type 2 diabetes, even after controlling for ba

--- Chunk 2 | ID: 120306 | Category: Symptoms | Distance: -13.6685 ---
Q: are patients with diabetes more likely to have atypical symptoms when seeking care of a first myocar
A: typical symptoms of myocardial infarction were equally prevalent in patients with and without diabetes and there were no sex differences in symptoms a

--- Chunk 3 | ID: 9306 | Category: Symptoms | Distance: -12.6262 ---
Q: is maternal diabetes distress linked to maternal depressive symptoms and adolescents ' glycemic cont
A: given the links between mothers' diabetes distress, maternal depressive symptoms, and adolescents' glyc

## 4. Test Full Pipeline (Single Query)

Full chain: embed → retrieve top-5 → build prompt → generate → add disclaimer.

In [4]:
result = pipeline.answer(test_query)

print(f"Question: {result['question']}")
print(f"\nAnswer:\n{result['answer']}")
print(f"\nDisclaimer present: {result['disclaimer_present']}")
print(f"Sources retrieved: {result['top_k']}")

Question: What are the symptoms of diabetes?

Answer:
functional health, self-rated health, and depressive symptoms in diabetes. ---

⚠️ MEDICAL DISCLAIMER: This response is generated by an AI system for educational and informational purposes only. It is NOT a substitute for professional medical advice, diagnosis, or treatment. Always consult a qualified healthcare provider for medical decisions.

Disclaimer present: True
Sources retrieved: 20


## 5. Test on 10 Sample Queries (with Logging)

The plan requires testing on 10 queries with logged context + generated answer.

In [5]:
test_queries = [
    # Symptoms
    "What are the early symptoms of type 2 diabetes?",
    # Diagnosis
    "How is pneumonia diagnosed in elderly patients?",
    # Treatment
    "What are the current treatment options for hypertension?",
    # Medication
    "What are the side effects of metformin?",
    # Prevention
    "How can cardiovascular disease be prevented through lifestyle changes?",
    # General
    "What is the role of the immune system in fighting infections?",
    # Mixed / clinical
    "Is laparoscopic surgery better than open surgery for prostatectomy?",
    "Can bacterial gastroenteritis lead to irritable bowel syndrome?",
    "Does naturopathy effectively treat menopausal symptoms?",
    "Is urgent colonoscopy needed for acute diverticular bleeding?",
]


In [6]:
results = []
latencies = []

print("=" * 100)
print("RUNNING 10-QUERY TEST")
print("=" * 100)

for i, query in enumerate(test_queries, 1):
    start = time.time()
    result = pipeline.answer(query)
    elapsed = (time.time() - start) * 1000
    latencies.append(elapsed)

    results.append({
        "query_num": i,
        "question": result["question"],
        "answer_raw": result["answer_raw"],
        "disclaimer_present": result["disclaimer_present"],
        "top_k": result["top_k"],
        "sources": result["retrieved_sources"],
        "latency_ms": round(elapsed, 2),
    })

    print(f"\n{'─' * 100}")
    print(f"Query {i}: {query}")
    print(f"⏱️  Latency: {elapsed:.0f}ms")
    print(f"📂 Sources: {[s['chunk_id'] for s in result['retrieved_sources']]}")
    print(f"📝 Answer: {result['answer_raw'][:200]}")
    print(f"⚠️  Disclaimer: {result['disclaimer_present']}")

RUNNING 10-QUERY TEST

────────────────────────────────────────────────────────────────────────────────────────────────────
Query 1: What are the early symptoms of type 2 diabetes?
⏱️  Latency: 1701ms
📂 Sources: [125695, 122314, 201425, 114356, 144426, 105602, 30752, 202936, 51164, 187324, 66662, 69529, 129317, 51877, 189938, 9306, 143666, 196494, 55845, 42715]
📝 Answer: The retrieved medical literature does not contain enough information to answer this question confidently. Please consult the listed sources directly or rephrase your question.
⚠️  Disclaimer: True

────────────────────────────────────────────────────────────────────────────────────────────────────
Query 2: How is pneumonia diagnosed in elderly patients?
⏱️  Latency: 1955ms
📂 Sources: [169585, 69972, 77178, 93285, 199421, 100023, 12480, 85513, 113358, 4515, 23070, 60564, 53796, 189220, 184990, 14962, 146434, 72166, 174187, 9346]
📝 Answer: Postoperative pneumonia is correlated with preoperative pulmonary obstructive dysf

## 6. Results Summary

In [7]:
print("\n" + "=" * 60)
print("📊 10-QUERY TEST SUMMARY")
print("=" * 60)

print(f"\nQueries tested:    {len(results)}")
print(f"All with disclaimer: {all(r['disclaimer_present'] for r in results)}")
print(f"All with sources:  {all(r['top_k'] >= 5 for r in results)}")

print(f"\n⏱️  Latency:")
print(f"  Min:    {min(latencies):.0f}ms")
print(f"  Max:    {max(latencies):.0f}ms")
print(f"  Mean:   {sum(latencies)/len(latencies):.0f}ms")

# Check disclaimer in every response
for r in results:
    assert r["disclaimer_present"], f"Missing disclaimer for query: {r['question']}"
print("\n✅ All 10 queries passed — disclaimer present, sources retrieved")


📊 10-QUERY TEST SUMMARY

Queries tested:    10
All with disclaimer: True
All with sources:  True

⏱️  Latency:
  Min:    1483ms
  Max:    3295ms
  Mean:   2227ms

✅ All 10 queries passed — disclaimer present, sources retrieved


## 7. Save Test Results Log

In [8]:
log_path = "../reports/rag_pipeline_test_log.json"
with open(log_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"✅ Test log saved to: {log_path}")

✅ Test log saved to: ../reports/rag_pipeline_test_log.json


## 8. Verify src/rag/pipeline.py Module

Confirm the module is importable and functional independently.

In [9]:
# Test that module-level convenience functions work
from src.rag.pipeline import answer as rag_answer

quick_result = rag_answer("What causes fever in children?")
print(f"Module-level answer() works: {len(quick_result['answer']) > 0}")
print(f"Answer preview: {quick_result['answer_raw'][:150]}")

Module-level answer() works: True
Answer preview: infections caused by usa300 isolates were associated with longer duration of fever. empirical treatment of sa should include mrsa. crp levels or 10 mg


## ✅ Summary

| Item | Status |
|---|---|
| LLM | `google/flan-t5-base` (free, local) |
| Retrieval | FAISS top-5 |
| Disclaimer | Appended to every response |
| Test queries | 10 queries logged with context + answer |
| `src/rag/pipeline.py` | Built and verified |

---

### ➡️ Next Step
Open **`07_classification_model.ipynb`** to fine-tune BioBERT classifier.